Packages


In [2]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.6 MB/s eta 0:00:00


Imports


In [3]:
import os
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 2 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
- SurendaranathK_Presentation.pdf


In [6]:
def load_pdf(pdf_path):
    """
    Extract text from a PDF.
    """
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    return text


documents = []

for pdf in pdf_files:

    text = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "text": text
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 2 document(s).


In [24]:
def chunk_text(text, chunk_size=1000, overlap=200):

    paragraphs = text.split("\n\n")

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) <= chunk_size:
            current_chunk += paragraph + "\n\n"
        else:
            chunks.append(current_chunk.strip())

            current_chunk = current_chunk[-overlap:] + paragraph + "\n\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [25]:
all_chunks = []

for document in documents:

    chunks = chunk_text(document["text"])

    for chunk in chunks:

        all_chunks.append(
            {
                "document": document["filename"],
                "text": chunk
            }
        )

print(f"Created {len(all_chunks)} chunks.")

Created 5 chunks.


In [26]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [27]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)
print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(5, 384)


In [30]:
import numpy as np

def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    similarities = np.dot(embeddings, query_embedding)

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "document": all_chunks[idx]["document"],
            "text": all_chunks[idx]["text"],
            "score": float(similarities[idx])
        })

    return results

In [31]:
results = retrieve("What algorithm was used for classification?", top_k=5)

for i, r in enumerate(results, 1):
    print("=" * 80)
    print(f"Result {i}")
    print(r["document"])
    print(r["text"][:800])

Result 1
SurendaranathK_Presentation.pdf
M.Sc - Artificial Intelligence and Machine Learning in Science
S P C 7 2 0 P  
R E S E A R C H  P R O J E C T  I N  D A T A  S C I E N C E  
Surendaranath Kanniyappan
 Supervisor : Dr. Michalis Agathos
MA C H I N E  L E A R N I N G  C L A S S I F I C A T I O N
O F  B I N A R Y  N E U T R O N  S T A R  R E MN A N T S
U S I N G  G R A V I T A T I O N A L  WA V E  D A T A
Result 2
SurendaranathK_Presentation.pdf
t curves, GRB energetics,
neutrinos).
Develop a low-latency pipeline for real-time GW alerts.
Extend framework to future GW detectors (Einstein Telescope, Cosmic Explorer).
FUTURE WORK
11
THANK YOU!
Result 3
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Machine Learning Classification of Binary Neutron
Star Remnants Using Gravitational Wave Data
Surendaranath Kanniyappan
Dr. Michalis Agathos
Abstract
Binary neutron star (BNS) mergers are among the most energetic cosmic events,
producing gravitational waves (GWs), electromagne

In [12]:
results = retrieve(
    "What is Retrieval Augmented Generation?"
)

for i, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"Result {i}")
    print("Document:", result["document"])
    print(result["text"])

Result 1
Document: Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
ovements.
• Number of estimators: chosen to minimise validation error without increasing variance.
• Early stopping: applied when validation performance plateaued.
GBDT was chosen because:
(a) it captures complex, non-linear decision boundaries;
(b) it performs well on moderate sized datasets without requiring vast training data;
(c) it offers interpretable feature importance metrics, aiding astrophysical validation.
8
3.6 SHAP Analysis
The SHapley Additive exPlanations (SHAP) [29] was used to i
Result 2
Document: Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
or remnants with lifetimes just above or below ∼ 5 ms.
From an astrophysical standpoint, the ability to classify outcomes with high MCC across all
three tasks has clear implications for multimessenger strategies: confident PCBH predictions
suggest negligible kilonova and GRB afterglow, while HMNS/NC outcomes increase the
likelihood of br

In [15]:
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="google/flan-t5-base"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

In [16]:
def ask(question):

    retrieved_chunks = retrieve(question)

    context = "\n\n".join(
        [chunk["text"] for chunk in retrieved_chunks]
    )

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=150
    )

    return response[0]["generated_text"]

In [17]:
question = "Summarize the main ideas from the uploaded documents."

answer = ask(question)

print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
ovements.
• Number of estimators: chosen to minimise validation error without increasing variance.
• Early stopping: applied when validation performance plateaued.
GBDT was chosen because:
(a) it captures complex, non-linear decision boundaries;
(b) it performs well on moderate sized datasets without requiring vast training data;
(c) it offers interpretable feature importance metrics, aiding astrophysical validation.
8
3.6 SHAP Analysis
The SHapley Additive exPlanations (SHAP) [29] was used to i

tween predictions and true labels across both majority and minority classes.
Feature importance and SHAP analysis: Global feature-importance analysis ranks ˜Λ as the
most influential parameter, followed by Mtot, q, and finally χeff. To gain a more interpretable,
sample level understanding, SHAP (SHapley Additive exPlanations) analysis is employed. The
SHAP summary plot in Figure 4 shows each feature’s distribution of contributions across all
validation samples. Positive SHAP values i

In [18]:
print(ask("What algorithm was used for classification?"))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
 (PCBH) and NS remnant. Classifier B: three-way
classification between PCBH, HMNS (short-lived and long-lived combined), and no collapse
(NC). Classifier C: four-way classification between PCBH, short-lived HMNS, long-lived
HMNS, and NC.
To minimise label uncertainty, any simulation classified as “no collapse” but with a postmerger
evolution timetsim < 25 mswas excluded [11]. Such cases may represent incomplete simulations
that ended before a collapse occurred, making the ground-truth label unre

enger campaigns.
21
A Assessing the probability of misclassification
In general, the uncertainty of predictions obtained through machine learning algorithms are
divided into two classes [33]:
• Irreducible or data uncertainty, caused by the complexity of the data, possible multi
modal features or noise. In classification problems, it is defined as the entropy of the
conditional probability of a given input x belonging to a class k:
H[p(y|x)] = −
KX
k=1
p(y = ωk | x) ln p(y = ωk | x),